In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.
Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online: not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json
from pydantic import BaseModel
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: list[str]

total_input_tokens = 0

for doc in documents[:3]:
	user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
      
	result, usage = llm_structured(
		openai_client,
		data_gen_instructions,
		user_prompt,
		Questions
	)
      
	total_input_tokens += usage.input_tokens

avg_input_tokens = total_input_tokens / 3
print(avg_input_tokens)

1353.0


In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

In [16]:
from tqdm.auto import tqdm
import numpy as np
from embedder import Embedder

embed = Embedder()

texts = [chunk["content"] for chunk in chunks]
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
	batch = texts[i:i+batch_size]
	batch_vectors = embed.encode_batch(batch)
	X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [17]:
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


In [21]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    v_query = embed.encode(query) 
    return vector_index.search(v_query, num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
            
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [26]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
q = ground_truth[0]["question"]

q2 = text_search(q)
print("q2: " +q2[0]['filename'])

q3 = vector_search(q)
print("q3: " + q3[0]['filename'])


q2: 01-agentic-rag/lessons/03-rag.md
q3: 01-agentic-rag/lessons/01-intro.md


In [27]:
def compute_relevance(q, search_function):
    target_filename = q["filename"]
    results = search_function(q["question"]) 

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == target_filename))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [28]:
q4 = evaluate(ground_truth, text_search)
print(f"Q4 - Hit Rate: {q4['hit_rate']:.2f}, MRR: {q4['mrr']:.2f}")

q5 = evaluate(ground_truth, vector_search)
print(f"Q5 - Hit Rate: {q5['hit_rate']:.2f}, MRR: {q5['mrr']:.2f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Q4 - Hit Rate: 0.76, MRR: 0.59


  0%|          | 0/360 [00:00<?, ?it/s]

Q5 - Hit Rate: 0.72, MRR: 0.55


In [29]:
k_values = [1, 50, 100, 200]

best_k = None
best_mrr = -1

for k in k_values:
	current_search_func = lambda query: hybrid_search(query, k=k)
	metrics = evaluate(ground_truth, current_search_func)
	current_mrr = metrics["mrr"]

	print(f"k ={k}: MRR = {current_mrr:.4f}\n")

	if current_mrr > best_mrr:
		best_mrr = current_mrr
		best_k = k

print(f"Q6 - Best k: {best_k}, Best MRR: {best_mrr:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

k =1: MRR = 0.6482



  0%|          | 0/360 [00:00<?, ?it/s]

k =50: MRR = 0.6379



  0%|          | 0/360 [00:00<?, ?it/s]

k =100: MRR = 0.6379



  0%|          | 0/360 [00:00<?, ?it/s]

k =200: MRR = 0.6379

Q6 - Best k: 1, Best MRR: 0.6482
